# Notebook 2: TF-IDF + Logistic Regression Baseline

**Research question:** Can a spam classifier trained on SMS messages generalize to emails?

This notebook establishes an interpretable baseline before neural modeling. Separate SMS and Enron classifiers are evaluated both in-domain and cross-domain.

## Experiment design

- Reuse the cleaned, leakage-free splits from Notebook 1.
- Represent text with word TF-IDF unigrams and bigrams, limited to 50,000 features.
- Train logistic regression with balanced class weights and fixed hyperparameters.
- Compare SMS -> SMS, SMS -> Enron, Enron -> SMS, and Enron -> Enron.

The first and last settings are in-domain; the other two are cross-domain. Spam F1 is the locked headline metric. Spam precision/recall, macro-F1, balanced accuracy, MCC, ROC-AUC, PR-AUC, class supports, and confusion counts provide complementary views of performance. No test data is used for model selection.

This fixed-configuration baseline is deterministic on the locked split, so repeated training seeds would reproduce the same estimate rather than provide a meaningful training-seed standard deviation. The repeated-seed protocol applies to the stochastic neural models; all models keep the same data partitions with split seed `42`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/kbozukov-coke/cross-domain-spam-detection.git'
PROJECT_NAME = 'cross-domain-spam-detection'
IS_KAGGLE = Path('/kaggle/working').exists()

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working') / PROJECT_NAME
    if not (PROJECT_ROOT / 'src').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_ROOT)],
            check=True,
        )
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

from src.data import load_prepared_splits, summarize_splits
from src.evaluation import build_prediction_table, stratified_bootstrap_ci
from src.modeling import run_transfer_experiments
from src.protocol import (
    BOOTSTRAP_SEED,
    DATA_SPLIT_SEED,
    DECISION_THRESHOLD,
    HEADLINE_METRIC,
    REFERENCE_TRAINING_SEED,
)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 120)
print(f'Running in: {"Kaggle" if IS_KAGGLE else "local environment"}')

## Load the prepared datasets

The same function as Notebook 1 downloads, cleans, splits, and validates both sources. No text from a test split is used for training.

In [ ]:
splits, _ = load_prepared_splits(random_state=DATA_SPLIT_SEED)
split_summary = summarize_splits(splits)
split_summary.loc[:, ['dataset', 'split', 'rows', 'ham', 'spam', 'spam_rate']]

## Train two baseline models

Each pipeline fits its vocabulary and classifier weights exclusively on its training domain. Balanced class weights compensate for the SMS class imbalance. Precision, recall, and F1 below explicitly use spam (label `1`) as the positive class; `tn`, `fp`, `fn`, and `tp` use the label order ham, spam.

In [ ]:
models, results = run_transfer_experiments(
    splits, random_state=REFERENCE_TRAINING_SEED
)

score_columns = [
    'accuracy', 'precision', 'recall', 'f1', 'macro_f1',
    'balanced_accuracy', 'mcc', 'roc_auc', 'pr_auc',
]
count_columns = ['ham_support', 'spam_support', 'tn', 'fp', 'fn', 'tp']
display_results = results.loc[
    :,
    [
        'train_domain', 'test_domain', 'setting', 'test_rows',
        *count_columns, *score_columns,
    ],
].rename(
    columns={
        'precision': 'spam_precision',
        'recall': 'spam_recall',
        'f1': 'spam_f1',
    }
).copy()
display_score_columns = [
    'accuracy', 'spam_precision', 'spam_recall', 'spam_f1', 'macro_f1',
    'balanced_accuracy', 'mcc', 'roc_auc', 'pr_auc',
]
display_results[display_score_columns] = (
    display_results[display_score_columns].round(3)
)
display_results

## Build stable per-example prediction tables

The already fitted models produce probabilities for each of the four train/test domain pairs. `build_prediction_table` assigns a content-derived `example_id` and the shared row schema used by later model notebooks, which makes paired comparisons and error analysis reproducible. This cell performs inference only; it does not refit either baseline.

In [ ]:
prediction_tables = {}
for train_domain in ('sms', 'enron'):
    for test_domain in ('sms', 'enron'):
        test_frame = splits[test_domain]['test']
        spam_probabilities = models[train_domain].predict_proba(
            test_frame['text']
        )[:, 1]
        prediction_tables[(train_domain, test_domain)] = build_prediction_table(
            test_frame,
            spam_probabilities,
            model='TF-IDF + Logistic Regression',
            training_seed=REFERENCE_TRAINING_SEED,
            train_domain=train_domain,
            test_domain=test_domain,
        )

baseline_predictions = pd.concat(
    prediction_tables.values(), ignore_index=True
)
prediction_inventory = (
    baseline_predictions.groupby(
        ['train_domain', 'test_domain', 'setting'], as_index=False
    )
    .agg(rows=('example_id', 'size'), unique_examples=('example_id', 'nunique'))
)
prediction_inventory

## Test-sample uncertainty for headline F1

For every domain pair, a stratified percentile bootstrap preserves the observed ham/spam counts and reports the individual spam-F1 estimate with a 95% confidence interval. The protocol locks `BOOTSTRAP_SEED`; 2,000 repetitions balance precision and notebook runtime.

These intervals quantify uncertainty from the finite test sample. They are not training-seed variability: this baseline is deterministic for the fixed data and configuration, so no training-seed mean or standard deviation is reported.

In [ ]:
N_BOOTSTRAP = 2_000
bootstrap_rows = []

for (train_domain, test_domain), predictions in prediction_tables.items():
    interval = stratified_bootstrap_ci(
        predictions['label'],
        predictions['spam_probability'],
        metric=HEADLINE_METRIC,
        n_bootstrap=N_BOOTSTRAP,
        confidence_level=0.95,
        random_state=BOOTSTRAP_SEED,
    )
    bootstrap_rows.append(
        {
            'train_domain': train_domain,
            'test_domain': test_domain,
            'setting': predictions['setting'].iat[0],
            **interval,
        }
    )

bootstrap_results = pd.DataFrame(bootstrap_rows)
bootstrap_display = bootstrap_results.loc[
    :,
    [
        'train_domain', 'test_domain', 'setting', 'metric', 'estimate',
        'ci_low', 'ci_high', 'bootstrap_standard_error',
        'confidence_level', 'n_bootstrap', 'random_state',
    ],
].rename(
    columns={
        'estimate': 'spam_f1_estimate',
        'ci_low': 'ci_95_low',
        'ci_high': 'ci_95_high',
    }
).copy()
interval_columns = [
    'spam_f1_estimate', 'ci_95_low', 'ci_95_high',
    'bootstrap_standard_error',
]
bootstrap_display[interval_columns] = (
    bootstrap_display[interval_columns].round(3)
)
bootstrap_display

## Compare in-domain and cross-domain F1

Holding the test domain fixed shows how performance changes with the training source. A large in-domain-to-cross-domain F1 drop indicates weak transfer.

In [ ]:
plt.figure(figsize=(8, 5))
axis = sns.barplot(data=results, x='test_domain', y='f1', hue='train_domain')
axis.set_title('Baseline F1 by training and test domain')
axis.set_xlabel('Test domain')
axis.set_ylabel('F1 score')
axis.set_ylim(0, 1)
axis.legend(title='Training domain')
for container in axis.containers:
    axis.bar_label(container, fmt='%.3f', padding=3)
plt.tight_layout()
plt.show()

## Confusion matrices

The matrices show whether transfer errors are dominated by missed spam or legitimate messages incorrectly flagged as spam.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for axis, row in zip(axes.ravel(), results.itertuples(index=False)):
    test_frame = splits[row.test_domain]['test']
    ConfusionMatrixDisplay.from_predictions(
        test_frame['label'],
        models[row.train_domain].predict(test_frame['text']),
        display_labels=['ham', 'spam'],
        colorbar=False,
        ax=axis,
    )
    axis.set_title(f'Train {row.train_domain.upper()} -> Test {row.test_domain.upper()}')

plt.tight_layout()
plt.show()

## Inspect SMS -> Enron errors

SMS -> Enron is the central transfer direction. We inspect high-confidence errors to identify systematic failures of the SMS-trained model on email.

In [ ]:
sms_to_enron = prediction_tables[('sms', 'enron')].copy()
cross_domain_errors = sms_to_enron.loc[~sms_to_enron['correct']].copy()
cross_domain_errors['error_type'] = cross_domain_errors['label'].map(
    {0: 'false positive', 1: 'false negative'}
)
cross_domain_errors['confidence'] = (
    cross_domain_errors['spam_probability'] - DECISION_THRESHOLD
).abs()

error_counts = cross_domain_errors['error_type'].value_counts().rename('errors')
display(error_counts.to_frame())

cross_domain_errors.sort_values('confidence', ascending=False).loc[
    :, ['error_type', 'spam_probability', 'text']
].head(5)

## Baseline conclusion

Each cross-domain point estimate is interpreted together with its test-sample confidence interval and the corresponding precision, recall, and confusion counts. The two transfer directions can fail differently, so the primary SMS -> Enron result should not be generalized from the reverse direction. The code below reports the observed primary transfer gap descriptively; the separate bootstrap intervals are not a confidence interval for that gap. Notebook 3 evaluates TextCNN under the same locked protocol.

In [ ]:
def select_interval(train_domain, test_domain):
    return bootstrap_results.query(
        'train_domain == @train_domain and test_domain == @test_domain'
    ).iloc[0]

sms_in_domain_interval = select_interval('sms', 'sms')
sms_to_enron_interval = select_interval('sms', 'enron')
transfer_gap = (
    sms_in_domain_interval['estimate'] - sms_to_enron_interval['estimate']
)

print(
    'SMS -> SMS spam F1: '
    f"{sms_in_domain_interval['estimate']:.3f} "
    f"(95% CI {sms_in_domain_interval['ci_low']:.3f}-"
    f"{sms_in_domain_interval['ci_high']:.3f})"
)
print(
    'SMS -> Enron spam F1: '
    f"{sms_to_enron_interval['estimate']:.3f} "
    f"(95% CI {sms_to_enron_interval['ci_low']:.3f}-"
    f"{sms_to_enron_interval['ci_high']:.3f})"
)
print(f'Observed primary transfer gap: {transfer_gap:.3f}')
print(
    'Interpret this descriptive gap with the full metrics, confusion counts, and '
    'finite-test-sample intervals above.'
)